<a href="https://colab.research.google.com/github/MyTeachers123/genai-course/blob/main/week2/Week_2_Lab2_Diffusion_DDPM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Diffusion Models: UNet DDPM from Scratch
**COM SCI-X 455 · Week 2 · Megan Armani, PhD**

> Push to `genai-course/week2/`

---

## Setup

In [ ]:
!pip install diffusers accelerate torch torchvision matplotlib --quiet

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", DEVICE)

## Note on Homework Extension
The **lab** trains on MNIST. Your **homework** extends to FashionMNIST:
```python
# One-line swap:
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
```
The UNet and training loop are identical. FashionMNIST is harder — observe how many epochs your model needs vs MNIST.

## Part 1 — Forward Process (Noise Schedule)
**Goal:** Implement the linear and cosine variance schedules and visualise what they do to an image.

In [ ]:
def linear_schedule(T=1000, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

def cosine_schedule(T=1000, s=0.008):
    steps = torch.arange(T+1, dtype=torch.float32)
    f = torch.cos(((steps/T + s) / (1+s)) * torch.pi/2) ** 2
    alphas_bar = f / f[0]
    betas = 1 - (alphas_bar[1:] / alphas_bar[:-1])
    return torch.clamp(betas, 0, 0.999)

In [ ]:
# TODO: Implement `q_sample(x0, t, betas)` — the closed-form forward process.
Formula: x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps
where eps ~ N(0,I) and alpha_bar_t = prod(1 - beta_i for i in 1..t)
Hint: precompute alpha_bar from betas.


In [ ]:
# TODO: Visualise the noising process: take one MNIST image, show it at t=0, 250, 500, 750, 1000
for BOTH linear and cosine schedules side by side.
Which schedule destroys fine detail faster?


## Part 2 — U-Net Noise Predictor

In [ ]:
class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-torch.arange(half, device=t.device) * (torch.log(torch.tensor(10000.0)) / (half-1)))
        args  = t[:, None].float() * freqs[None]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, ch, time_dim):
        super().__init__()
        self.conv1  = nn.Conv2d(ch, ch, 3, padding=1)
        self.conv2  = nn.Conv2d(ch, ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, ch)
        self.norm1  = nn.GroupNorm(8, ch)
        self.norm2  = nn.GroupNorm(8, ch)
    def forward(self, x, t_emb):
        h = self.norm1(x)
        h = torch.relu(self.conv1(h))
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.norm2(h)
        return x + self.conv2(torch.relu(h))

In [ ]:
# TODO: Build a UNet class with:
- Encoder: 3 levels (32→64→128 channels), each with ResBlock + downsample
- Bottleneck: ResBlock with time embedding
- Decoder: 3 levels with skip connections + upsample
- Time embedding fed to every ResBlock
Input: (B, 1, 28, 28) noisy image + timestep t
Output: (B, 1, 28, 28) predicted noise


## Part 3 — Training Loop

In [ ]:
# TODO: Train your UNet DDPM for 20 epochs on MNIST.
Loss: MSE between true noise eps and predicted noise eps_theta(x_t, t)
Use Adam lr=1e-3, batch_size=128.
Plot training loss curve.


## Part 4 — DDIM vs DDPM Sampling

In [ ]:
# DDPM sampling (1000 steps — slow)
@torch.no_grad()
def ddpm_sample(model, betas, T=1000, shape=(1,1,28,28)):
    alphas     = 1 - betas
    alpha_bars = torch.cumprod(alphas, dim=0).to(DEVICE)
    x = torch.randn(shape).to(DEVICE)
    for t in reversed(range(T)):
        t_tensor = torch.tensor([t]).to(DEVICE)
        eps_pred = model(x, t_tensor)
        alpha_bar_t = alpha_bars[t]
        alpha_bar_prev = alpha_bars[t-1] if t > 0 else torch.tensor(1.0)
        # posterior mean
        coef1 = betas[t] / (1 - alpha_bar_t).sqrt()
        x = (1/alphas[t].sqrt()) * (x - coef1 * eps_pred)
        if t > 0:
            noise = torch.randn_like(x)
            x = x + betas[t].sqrt() * noise
    return x

In [ ]:
# TODO: Implement `ddim_sample(model, betas, steps=50)` — deterministic, skip timesteps.
Time 100 DDPM samples vs 100 DDIM samples.
Display side-by-side quality comparison. What is the FID difference?


### ❓ Reflect
1. Did cosine schedule produce better results than linear at 20 epochs? What visual difference did you see?
2. How much faster was DDIM vs DDPM? Was the quality difference acceptable for your use case?
3. Why does DDIM produce the same output every run given the same starting noise?

In [ ]:
# Reflection


---
## Submission
- [ ] Push to `genai-course/week2/lab2.ipynb`
- [ ] Loss curve plotted
- [ ] Schedule comparison visualised
- [ ] DDIM vs DDPM timing + quality comparison